In [ ]:

from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_redis import RedisConfig, RedisVectorStore,RedisConfig
from langchain_core.documents import Document
from typing import Set, List, Any, Dict, Union, Optional
import numpy as np
from dotenv import load_dotenv
import redis
import os
load_dotenv()
REDIS_URL = os.getenv("REDIS_URL")
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
redis_client = redis.Redis.from_url(REDIS_URL, decode_responses=False) 
nameofindex = "trial_skills_embedding"
config = RedisConfig(
      index_name=nameofindex,
      legacy_key_format=False,
      distance_metric="L2" #euclidian
)
vectorstore = RedisVectorStore(
    embeddings=embeddings,
    config=config,
    
)

ConnectionError: Error 10061 connecting to localhost:6379. No connection could be made because the target machine actively refused it.

In [ ]:
employees = ['Emp_A', 'Emp_B', 'Emp_C', 'Emp_D','Emp_E'] # Set I
alpha = 0.8
number_of_k = 5
distances={}
employee_skills = [
  [
    "Python",
    "SQL (PostgreSQL/MySQL)",
    "Cloud Infrastructure (AWS/Azure)",
    "Docker/Kubernetes",
    "System Architecture Design",
    "Agile/Scrum Mastery"
],
 [
    "JavaScript (React/Vue)",
    "HTML5/CSS3 (Tailwind CSS)",
    "API Integration (REST/GraphQL)",
    "Unit Testing (Jest/Enzyme)",
    "Responsive Design",
    "User Empathy",
    "A/B Testing Analysis"
],
 [
    "R/Pandas/NumPy",
    "Data Warehousing (Snowflake)",
    "ETL Pipeline Development",
    "Data Visualization (Tableau/Power BI)",
    "Critical Thinking",
    "Inference and Reporting"
],
  [
    "Network Security (Firewalls, IDS/IPS)",
    "Penetration Testing (Kali Linux)",
    "Security Information and Event Management (SIEM)",
    "Shell Scripting (Bash/PowerShell)",
    "Incident Response",
    "Risk Assessment",
    "Pressure Management"
],
  [
    "Terraform/Ansible",
    "Continuous Integration/Continuous Deployment (CI/CD)",
    "Linux Administration",
    "Monitoring Tools (Prometheus/Grafana)",
    "Version Control (Git)",
    "Time Management"
]
 
  
]

requirements =[
    "Cloud Security Posture Management (CSPM)",
    "Identity and Access Management (IAM)",
    "Kubernetes Security",
    "Security as Code (SaC)",
    "Network Segmentation (VPC/VNet)",
    "Threat Modeling",
    "Policy Development",
] #set J

In [ ]:
import datetime
i = 0
for employee in employees:
    for skills in employee_skills[i]:
        doc = Document(
                    page_content=skills,
                    metadata={
                        "employee": employee,
                        "timestamp": datetime.now().strftime("%Y%m%d_%H%M%S")
                    }
                )
        vectorstore.add_documents(doc)
    i+=1

In [ ]:
def calculate_cosine_similarity(vec1: List[float], vec2: List[float]) -> float:
    """Menghitung Cosine Similarity antara dua vektor."""
    vec1 = np.array(vec1).flatten()
    vec2 = np.array(vec2).flatten()
    dot_product = np.dot(vec1, vec2)
    norm_a = np.linalg.norm(vec1)
    norm_b = np.linalg.norm(vec2)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    similarity = dot_product / (norm_a * norm_b)
    return float(similarity)

# for skills in requirements:
#   actual_embedding = embeddings.embed_query(skills)
#   distance[vectorstore.similarity_search_by_vector(actual_embedding, k=number_of_k)

In [ ]:
import pyomo.environ as pe
import pyomo.opt as po

solver = po.SolverFactory('glpk')

In [ ]:
import pyomo.environ as pyo
import numpy as np

# ==========================================
# 1. SETUP DUMMY DATA (Simulating Embeddings)
# ==========================================
# In reality, you calculate these distances using:
# scipy.spatial.distance.cosine(employee_vec, task_vec)


# Simulating the Distance Matrix (d_ij)
# Lower number = Better match (less distance)
# Format: {(Employee, Task): Distance}

# ==========================================
# 2. THE PYOMO MODEL
# ==========================================

model = pyo.ConcreteModel()

# --- Sets ---
model.I = pyo.Set(initialize=employees)    # Employees
model.J = pyo.Set(initialize=requirements) # Requirements

# --- Parameters ---
model.d = pyo.Param(model.I, model.J, initialize=distances) # d_ij
model.M = pyo.Param(initialize=len(requirements)) # Big-M (Total tasks)

# --- Weights for Multi-Objective ---
# Alpha: Priority for Quality (Distance)
# Beta: Priority for Saving Headcount (Cost)
S = len(requirements)/len(employees)

beta = 1-alpha * S   

# Instead of a full matrix, we have a Dictionary of Lists
# {Task: [(Emp_A, 0.1), (Emp_B, 0.15)... top 5]}
top_k_data = {}

for r in requirements:
    # Randomly pick 5 employees to be the "closest vectors" for this task
    candidates = random.sample(all_employees, 5)
    candidates_with_scores = []
    for emp in candidates:
        # L2 Distance (lower is better)
        dist = round(random.uniform(0.1, 0.8), 2)
        candidates_with_scores.append((emp, dist))
    top_k_data[r] = candidates_with_scores

# ==========================================
# 2. PRE-PROCESSING FOR PYOMO
# ==========================================
# We need to flatten the data into a list of valid (Emp, Task) pairs
valid_assignments = [] 
distance_map = {}

for task, candidates in top_k_data.items():
    for emp, dist in candidates:
        valid_assignments.append((emp, task))
        distance_map[(emp, task)] = dist

# ==========================================
# 3. PYOMO MODEL (SPARSE)
# ==========================================
model = pyo.ConcreteModel()

# --- Sets ---
model.I = pyo.Set(initialize=all_employees) # All Employees (for y_i)
model.J = pyo.Set(initialize=requirements)  # All Tasks

# **CRITICAL CHANGE**: The set of allowed connections
# dimen=2 means it stores pairs like ('Emp_1', 'Task_1')
model.ValidPairs = pyo.Set(initialize=valid_assignments, dimen=2)

# --- Parameters ---
# Map the (Emp, Task) tuple to the distance
model.d = pyo.Param(model.ValidPairs, initialize=distance_map)

# Max tasks one person can reasonably take (Optimization bound)
model.M = pyo.Param(initialize=len(requirements))

# --- Variables ---
# p is now indexed ONLY by ValidPairs (Sparse Variable)
model.p = pyo.Var(model.ValidPairs, domain=pyo.Binary)
model.y = pyo.Var(model.I, domain=pyo.Binary)

# --- Weights ---
alpha = 1.0  # Quality
beta = 5.0   # Headcount Cost

# --- Objective ---
def objective_rule(model):
    # Sum only over the valid pairs
    quality = sum(model.d[i, j] * model.p[i, j] for (i, j) in model.ValidPairs)
    headcount = sum(model.y[i] for i in model.I)
    return (alpha * quality) + (beta * headcount)

model.Obj = pyo.Objective(rule=objective_rule, sense=pyo.minimize)

# --- Constraints ---

# 1. Every task must be assigned to exactly one employee (from its top 5)
def one_employee_rule(model, j):
    # Find all employees 'i' capable of doing task 'j' (i.e., exists in ValidPairs)
    # This filter simulates "i in N(j)"
    candidates_for_j = [i for i in model.I if (i, j) in model.ValidPairs]
    return sum(model.p[i, j] for i in candidates_for_j) == 1

model.AssignOne = pyo.Constraint(model.J, rule=one_employee_rule)

# 2. Link p[i,j] to y[i] (Activation)
def activation_rule(model, i):
    # Find all tasks 'j' that employee 'i' is eligible for
    tasks_for_i = [j for j in model.J if (i, j) in model.ValidPairs]
    
    # If the employee has NO eligible tasks in the top 5 lists, force y[i] to 0
    if not tasks_for_i:
        return model.y[i] == 0
        
    return sum(model.p[i, j] for j in tasks_for_i) <= model.M * model.y[i]

model.ActivateEmp = pyo.Constraint(model.I, rule=activation_rule)

# ==========================================
# 4. SOLVE
# ==========================================
solver = pyo.SolverFactory('glpk') # or 'cbc'
results = solver.solve(model)

print("\n--- RESULTS ---")
for i in model.I:
    if pyo.value(model.y[i]) > 0.5:
        print(f"\nEmployee {i} is Active:")
        for j in model.J:
            # Check if (i,j) is a valid pair before asking for value
            if (i, j) in model.ValidPairs and pyo.value(model.p[i, j]) > 0.5:
                print(f"  -> Assigned {j} (Distance: {distance_map[(i,j)]})")


--- OPTIMIZATION RESULTS ---
Total Objective Cost: 1.86

Employee Emp_C is ACTIVE:
  --> Assigned: Task_1 (Dist: 0.12)
  --> Assigned: Task_4 (Dist: 0.29)
  --> Assigned: Task_5 (Dist: 0.26)

Employee Emp_G is ACTIVE:
  --> Assigned: Task_2 (Dist: 0.25)
  --> Assigned: Task_3 (Dist: 0.16)

Total Employees Used: 2 / 8


In [ ]:
import pyomo.environ as pyo
import numpy as np

# ==========================================
# 1. SETUP DUMMY DATA (Simulating Embeddings)
# ==========================================
# In reality, you calculate these distances using:
# scipy.spatial.distance.cosine(employee_vec, task_vec)

employees = ['Emp_A', 'Emp_B', 'Emp_C', 'Emp_D','Emp_E'] # Set I
requirements = ['Task_1', 'Task_2', 'Task_3', 'Task_4', 'Task_5'] # Set J
alpha = 0.8
# Simulating the Distance Matrix (d_ij)
# Lower number = Better match (less distance)
# Format: {(Employee, Task): Distance}
distances = {}
np.random.seed(42) # Fixed seed for reproducibility
for e in employees:
    for r in requirements:
        # Random distance between 0.1 (good fit) and 1.0 (bad fit)
        distances[(e, r)] = np.round(np.random.uniform(0.1, 1.0), 2)

# ==========================================
# 2. THE PYOMO MODEL
# ==========================================

model = pyo.ConcreteModel()

# --- Sets ---
model.I = pyo.Set(initialize=employees)    # Employees
model.J = pyo.Set(initialize=requirements) # Requirements

# --- Parameters ---
model.d = pyo.Param(model.I, model.J, initialize=distances) # d_ij
model.M = pyo.Param(initialize=len(requirements)) # Big-M (Total tasks)

# --- Weights for Multi-Objective ---
# Alpha: Priority for Quality (Distance)
# Beta: Priority for Saving Headcount (Cost)
S = len(requirements)/len(employees)

beta = 1-alpha * S   

# --- Variables ---
# p[i,j]: 1 if employee i takes task j
model.p = pyo.Var(model.I, model.J, domain=pyo.Binary)

# y[i]: 1 if employee i is active (has at least 1 task)
model.y = pyo.Var(model.I, domain=pyo.Binary)

# --- Objective Function ---
# Minimize: (Alpha * Total Distance) + (Beta * Active Employees)
def objective_rule(model):
    quality_score = sum(model.d[i, j] * model.p[i, j] for i in model.I for j in model.J)
    headcount_score = sum(model.y[i] for i in model.I)
    return (alpha * quality_score) + (beta * headcount_score)

model.Obj = pyo.Objective(rule=objective_rule, sense=pyo.minimize)

# --- Constraints ---

# 1. Single Assignment Constraint
# Each task (j) must be assigned to exactly one employee
def one_employee_per_task(model, j):
    return sum(model.p[i, j] for i in model.I) == 1

model.Constraint1 = pyo.Constraint(model.J, rule=one_employee_per_task)

# 2. Linking Constraint (Big-M)
# If an employee takes tasks, y[i] must be 1.
# Sum(tasks) <= M * y[i]
def link_employee_active(model, i):
    return sum(model.p[i, j] for j in model.J) <= model.M * model.y[i]

model.Constraint2 = pyo.Constraint(model.I, rule=link_employee_active)

# ==========================================
# 3. SOLVE AND PRINT
# ==========================================

# You need 'glpk', 'cbc', or 'gurobi' installed
solver = pyo.SolverFactory('glpk') 

try:
    results = solver.solve(model)
    
    print("\n--- OPTIMIZATION RESULTS ---")
    print(f"Total Objective Cost: {pyo.value(model.Obj):.2f}")
    
    active_employees = []
    
    for i in model.I:
        # Check if employee is active
        if pyo.value(model.y[i]) > 0.5:
            active_employees.append(i)
            print(f"\nEmployee {i} is ACTIVE:")
            for j in model.J:
                if pyo.value(model.p[i, j]) > 0.5:
                    dist = distances[(i,j)]
                    print(f"  --> Assigned: {j} (Dist: {dist})")
    
    print(f"\nTotal Employees Used: {len(active_employees)} / {len(employees)}")

except Exception as e:
    print("Solver error! Do you have a solver installed?")
    print(e)

In [12]:
import pyomo.environ as pyo
import random

# ==========================================
# 1. MOCK DATA (Top 5 Matches Only)
# ==========================================
all_employees = [f'Emp_{i}' for i in range(1, 21)] # 20 Employees
requirements = [f'Task_{j}' for j in range(1, 11)] # 10 Tasks

# Instead of a full matrix, we have a Dictionary of Lists
# {Task: [(Emp_A, 0.1), (Emp_B, 0.15)... top 5]}
top_k_data = {}

for r in requirements:
    # Randomly pick 5 employees to be the "closest vectors" for this task
    candidates = random.sample(all_employees, 5)
    candidates_with_scores = []
    for emp in candidates:
        # L2 Distance (lower is better)
        dist = round(random.uniform(0.1, 0.8), 2)
        candidates_with_scores.append((emp, dist))
    top_k_data[r] = candidates_with_scores

# ==========================================
# 2. PRE-PROCESSING FOR PYOMO
# ==========================================
# We need to flatten the data into a list of valid (Emp, Task) pairs
valid_assignments = [] 
distance_map = {}

for task, candidates in top_k_data.items():
    for emp, dist in candidates:
        valid_assignments.append((emp, task))
        distance_map[(emp, task)] = dist

# ==========================================
# 3. PYOMO MODEL (SPARSE)
# ==========================================
model = pyo.ConcreteModel()

# --- Sets ---
model.I = pyo.Set(initialize=all_employees) # All Employees (for y_i)
model.J = pyo.Set(initialize=requirements)  # All Tasks

# **CRITICAL CHANGE**: The set of allowed connections
# dimen=2 means it stores pairs like ('Emp_1', 'Task_1')
model.ValidPairs = pyo.Set(initialize=valid_assignments, dimen=2)

# --- Parameters ---
# Map the (Emp, Task) tuple to the distance
model.d = pyo.Param(model.ValidPairs, initialize=distance_map)

# Max tasks one person can reasonably take (Optimization bound)
model.M = pyo.Param(initialize=len(requirements))

# --- Variables ---
# p is now indexed ONLY by ValidPairs (Sparse Variable)
model.p = pyo.Var(model.ValidPairs, domain=pyo.Binary)
model.y = pyo.Var(model.I, domain=pyo.Binary)

# --- Weights ---
alpha = 1.0  # Quality
beta = 5.0   # Headcount Cost

# --- Objective ---
def objective_rule(model):
    # Sum only over the valid pairs
    quality = sum(model.d[i, j] * model.p[i, j] for (i, j) in model.ValidPairs)
    headcount = sum(model.y[i] for i in model.I)
    return (alpha * quality) + (beta * headcount)

model.Obj = pyo.Objective(rule=objective_rule, sense=pyo.minimize)

# --- Constraints ---

# 1. Every task must be assigned to exactly one employee (from its top 5)
def one_employee_rule(model, j):
    # Find all employees 'i' capable of doing task 'j' (i.e., exists in ValidPairs)
    # This filter simulates "i in N(j)"
    candidates_for_j = [i for i in model.I if (i, j) in model.ValidPairs]
    return sum(model.p[i, j] for i in candidates_for_j) == 1

model.AssignOne = pyo.Constraint(model.J, rule=one_employee_rule)

# 2. Link p[i,j] to y[i] (Activation)
def activation_rule(model, i):
    # Find all tasks 'j' that employee 'i' is eligible for
    tasks_for_i = [j for j in model.J if (i, j) in model.ValidPairs]
    
    # If the employee has NO eligible tasks in the top 5 lists, force y[i] to 0
    if not tasks_for_i:
        return model.y[i] == 0
        
    return sum(model.p[i, j] for j in tasks_for_i) <= model.M * model.y[i]

model.ActivateEmp = pyo.Constraint(model.I, rule=activation_rule)

# ==========================================
# 4. SOLVE
# ==========================================
solver = pyo.SolverFactory('glpk') # or 'cbc'
results = solver.solve(model)

print("\n--- RESULTS ---")
for i in model.I:
    if pyo.value(model.y[i]) > 0.5:
        print(f"\nEmployee {i} is Active:")
        for j in model.J:
            # Check if (i,j) is a valid pair before asking for value
            if (i, j) in model.ValidPairs and pyo.value(model.p[i, j]) > 0.5:
                print(f"  -> Assigned {j} (Distance: {distance_map[(i,j)]})")


--- RESULTS ---

Employee Emp_5 is Active:
  -> Assigned Task_1 (Distance: 0.16)
  -> Assigned Task_8 (Distance: 0.34)

Employee Emp_14 is Active:
  -> Assigned Task_2 (Distance: 0.29)
  -> Assigned Task_3 (Distance: 0.14)
  -> Assigned Task_5 (Distance: 0.5)

Employee Emp_17 is Active:
  -> Assigned Task_4 (Distance: 0.53)
  -> Assigned Task_6 (Distance: 0.44)
  -> Assigned Task_7 (Distance: 0.41)
  -> Assigned Task_9 (Distance: 0.5)
  -> Assigned Task_10 (Distance: 0.46)
